In [1]:
from docopt import docopt
import pandas as pd
import mudata as mdata
import anndata as ad
# Custom import fun
# from preprocess_view import preprocess_view
# from convert_binary_str import convert_binary_str

In [3]:
raw_data = mdata.read("tcga-blca.h5mu")

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/mudata/_core/mudata.py:491: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(


In [8]:
data.obs

,Methylation_Gene_level_HM450K:age,Methylation_Gene_level_HM450K:sex,Methylation_Gene_level_HM450K:response,miRNA_Gene_level:age,miRNA_Gene_level:sex,miRNA_Gene_level:response,RPPA_Gene_Level:age,RPPA_Gene_Level:sex,RPPA_Gene_Level:response,RNAseq_HiSeq_Gene_level:age,RNAseq_HiSeq_Gene_level:sex,RNAseq_HiSeq_Gene_level:response
sample_name,,,,,,,,,,,,
TCGA.2F.A9KT,83,1.0,no,83,1.0,no,83,1.0,no,83,1.0,no
TCGA.4Z.AA7M,65,1.0,yes,65,1.0,yes,65,1.0,yes,65,1.0,yes
TCGA.4Z.AA7Q,79,1.0,yes,79,1.0,yes,79,1.0,yes,79,1.0,yes
TCGA.4Z.AA84,61,1.0,yes,61,1.0,yes,61,1.0,yes,61,1.0,yes
TCGA.4Z.AA89,60,1.0,yes,60,1.0,yes,60,1.0,yes,60,1.0,yes
...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA.ZF.AA52,70,1.0,yes,70,1.0,yes,70,1.0,yes,70,1.0,yes
TCGA.ZF.AA54,71,1.0,yes,71,1.0,yes,71,1.0,yes,71,1.0,yes
TCGA.ZF.AA56,79,0.0,yes,79,0.0,yes,79,0.0,yes,79,0.0,yes


In [7]:
data = raw_data.copy() 
# Mudata has stricter format, so only need to check if each obs contain the
# required columns
accepted_cols = set(["response", "sample_names", "sample_name"])
new_mu_dict = {}
for modality in data.mod:
    # get each modality's obs metadata and the measurements
    omic_obs = data[modality].obs
    obs_cols = set(omic_obs.columns)
    print(obs_cols)
    # Check if at least contains response or sample_name/s in accepted cols
    cols_contained = len(accepted_cols & obs_cols) >= 2
    if not cols_contained:
        raise Exception("Did not have the contained columns: response, sample_name")

{'age', 'response', 'sex'}


/usr/local/lib/python3.10/dist-packages/mudata/_core/mudata.py:491: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(


Exception: Did not have the contained columns: response, sample_name

In [13]:
data.mod

{'Methylation_Gene_level_HM450K': AnnData object with n_obs × n_vars = 336 × 8513
     obs: 'age', 'sex', 'response'
     var: 'feature',
 'miRNA_Gene_level': AnnData object with n_obs × n_vars = 336 × 520
     obs: 'age', 'sex', 'response'
     var: 'feature',
 'RPPA_Gene_Level': AnnData object with n_obs × n_vars = 336 × 55
     obs: 'age', 'sex', 'response'
     var: 'feature',
 'RNAseq_HiSeq_Gene_level': AnnData object with n_obs × n_vars = 336 × 18845
     obs: 'age', 'sex', 'response'
     var: 'feature'}

In [14]:
data["Methylation_Gene_level_HM450K"].obs.index.name in accepted_cols

True

In [ ]:

# # TODO: NOT scaling now, since it might introduce negative numbers and cause problem


# def transform_mudata_format(mu_path, dataset_name, identifier_col="sample_name", var_threshold=0.16, replace_na_val=0, scale=False):
#     # Read data here
#     raw_data = mdata.read(mu_path)
#     # Use a deeper copy of it
#     data = raw_data.copy() 
    # Mudata has stricter format, so only need to check if each obs contain the
    # required columns
    accepted_cols = set(["response", "sample_names", "sample_name"])
    new_mu_dict = {}
    for modality in data.mod:
        # get each modality's obs metadata and the measurements
        omic_obs = data[modality].obs
        obs_cols = set(omic_obs.columns)
        # Check if at least contains response or sample_name/s in accepted cols
        cols_contained = len(accepted_cols & obs_cols) >= 2 or omic_obs.index.name in accepted_cols
        if not cols_contained:
            raise Exception("Did not have the contained columns: response, sample_name")
        
#         # TODO: Uggly fix now to add the sample_name into it
#         # Check if identifier_col is already in the DataFrame
#         if identifier_col in omic_obs.columns:
#             # If identifier_col is present, drop sample_names if it exists
#             if 'sample_names' in omic_obs.columns:
#                 omic_obs = omic_obs.drop(columns=['sample_names'])
#         else:
#             # If identifier_col is not present, check if sample_names is in the DataFrame
#             if 'sample_names' in omic_obs.columns:
#                 # If sample_names is present, rename it to identifier_col
#                 omic_obs = omic_obs.rename(columns={'sample_names': identifier_col})
#             else:
#                 # If neither are present, create identifier_col from other sources
#                 omic_obs[identifier_col] = data.obs_names.to_list()  # Replace with your logic for creating identifier_col
#         # And coerce the index of this to sample name as well
#         omic_obs.index.name = identifier_col            
#         # Then check type of the reponse and convert it string only
#         omic_obs["response"] = convert_binary_str(omic_obs["response"])

#         # Then start the preprocess steps
#         omic_df = data[modality].to_df()
#         # Remove those of near zero variance
#         # And replace nas with 0
#         # And scale each
#         df_reduced = preprocess_view(df = omic_df, var_threshold = var_threshold, replace_na_val=replace_na_val, scale=scale)
#         # Recreate new AnnData
#         new_ann = ad.AnnData(X = df_reduced, 
#                              obs=omic_obs, 
#                              var=pd.DataFrame(df_reduced.columns, 
#                                               columns=["feature"], 
#                                               index=df_reduced.columns)
#                             )
#         new_mu_dict[modality] = new_ann
#     # Then create a new mudata object
#     new_mdata = mdata.MuData(new_mu_dict)
#     # Lastly just write it to disk
#     output_name = dataset_name + ".h5mu"
#     # This data is a MuData object, hence could access its write method
#     new_mdata.write(output_name)
#     return new_mdata

# # Execute the fun here
# if __name__ == '__main__':
#   # Parse docopt
#   args = docopt(__doc__)
#   # Execute runner
#   transform_mudata_format(
#     mu_path=args['--mu_path'] , dataset_name=args['--dataset_name'], 
#     var_threshold=float(args['--var_threshold']), replace_na_val=float(args['--replace_na_val'])
#     )